In [1]:
import torch
import time
import numpy as np
from sparse_transformers import WeightCache

Configuring for 28 CPU threads


In [2]:

def profile_baseline(gate_weight, up_weight, down_weight, masks, hidden_dim, device):
    """Profile baseline approach using torch indexing"""
    times = []
    x = torch.randn(1, hidden_dim, device=device)

    for mask in masks:
        start = time.perf_counter()

        # Index into weights
        active_gate = gate_weight[mask]
        active_up = up_weight[mask]
        active_down = down_weight[:, mask]

        # Concat gate+up
        active_weights = torch.cat([active_gate, active_up], dim=0)

        # Forward pass
        gate, up = x.matmul(active_weights.t()).chunk(2, dim=-1)
        out = (torch.nn.functional.silu(gate) * up).matmul(active_down.t())

        torch.cuda.synchronize() if device.type == 'cuda' else None
        times.append(time.perf_counter() - start)

    return times

def profile_weight_cache(gate_weight, up_weight, down_weight, masks, hidden_dim, device):
    """Profile weight cache approach"""
    times = []
    x = torch.randn(1, hidden_dim, device=device)

    # Initialize cache with first mask
    cache = WeightCache(masks[0], hidden_dim, gate_weight, up_weight, down_weight, True)

    for mask in masks:
        start = time.perf_counter()

        # Update cache (differential update)
        cache.update_active_weights(mask)

        # Forward pass
        gate, up = x.matmul(cache.get_concat_weight().t()).chunk(2, dim=-1)
        out = (torch.nn.functional.silu(gate) * up).matmul(cache.get_active_down_weight().t())

        torch.cuda.synchronize() if device.type == 'cuda' else None
        times.append(time.perf_counter() - start)

    return times

def generate_masks(n_masks, intermediate_size, n_active, n_swapped):
    """Generate sequence of masks where n_swapped indices change each step"""
    masks = []

    # Initial mask
    mask = torch.zeros(intermediate_size, dtype=torch.bool)
    active_indices = torch.randperm(intermediate_size)[:n_active]
    mask[active_indices] = True
    masks.append(mask.clone())

    # Generate subsequent masks by swapping indices
    active_set = set(active_indices.tolist())
    inactive_set = set(range(intermediate_size)) - active_set

    for _ in range(n_masks - 1):
        # Randomly select indices to swap
        to_remove = np.random.choice(list(active_set), size=min(n_swapped, len(active_set)), replace=False)
        to_add = np.random.choice(list(inactive_set), size=min(n_swapped, len(inactive_set)), replace=False)

        # Update sets
        active_set -= set(to_remove)
        active_set |= set(to_add)
        inactive_set -= set(to_add)
        inactive_set |= set(to_remove)

        # Create new mask
        mask = torch.zeros(intermediate_size, dtype=torch.bool)
        mask[list(active_set)] = True
        masks.append(mask.clone())

    return masks

In [ ]:
intermediate_size = 8192
hidden_dim = 3072
n_active = int(intermediate_size * 0.5)  # 50% sparsity
n_swapped = int(n_active * 0.1)  # Swap 10% of active indices each step
n_runs = 100

device=torch.device('cpu')

masks = generate_masks(n_runs, intermediate_size, n_active, n_swapped)

gate_weight = torch.randn(intermediate_size, hidden_dim, device=device)
up_weight = torch.randn(intermediate_size, hidden_dim, device=device)
down_weight = torch.randn(hidden_dim, intermediate_size, device=device)

In [3]:

def run_profile(intermediate_size, hidden_dim, n_active, n_swapped, n_runs, device='cpu'):
    print(f"\nProfiling Configuration:")
    print(f"  Intermediate size: {intermediate_size}")
    print(f"  Hidden dim: {hidden_dim}")
    print(f"  Active indices: {n_active} ({100*n_active/intermediate_size:.1f}%)")
    print(f"  Swapped per step: {n_swapped} ({100*n_swapped/n_active:.1f}% of active)")
    print(f"  Number of runs: {n_runs}")
    print(f"  Device: {device}")

    # Create weights
    gate_weight = torch.randn(intermediate_size, hidden_dim, device=device)
    up_weight = torch.randn(intermediate_size, hidden_dim, device=device)
    down_weight = torch.randn(hidden_dim, intermediate_size, device=device)

    # Generate masks
    masks = generate_masks(n_runs, intermediate_size, n_active, n_swapped)
    masks = [m.to(device) for m in masks]

    # Warm up
    print("\nWarming up...")
    _ = profile_baseline(gate_weight, up_weight, down_weight, masks[:5], hidden_dim, device)
    _ = profile_weight_cache(gate_weight, up_weight, down_weight, masks[:5], hidden_dim, device)

    # Profile baseline
    print("Profiling baseline...")
    baseline_times = profile_baseline(gate_weight, up_weight, down_weight, masks, hidden_dim, device)

    # Profile weight cache
    print("Profiling weight cache...")
    cache_times = profile_weight_cache(gate_weight, up_weight, down_weight, masks, hidden_dim, device)

    # Results
    baseline_avg = np.mean(baseline_times[5:]) * 1000  # Skip first 5, convert to ms
    cache_avg = np.mean(cache_times[5:]) * 1000
    speedup = baseline_avg / cache_avg

    print(f"\nResults:")
    print(f"  Baseline avg: {baseline_avg:.3f} ms")
    print(f"  Cache avg: {cache_avg:.3f} ms")
    print(f"  Speedup: {speedup:.2f}x")

    return {
        'baseline_avg': baseline_avg,
        'cache_avg': cache_avg,
        'speedup': speedup
    }

if __name__ == '__main__':
    # Default parameters for Llama 3.2 3B
    intermediate_size = 8192
    hidden_dim = 3072
    n_active = int(intermediate_size * 0.5)  # 50% sparsity
    n_swapped = int(n_active * 0.1)  # Swap 10% of active indices each step
    n_runs = 100

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    run_profile(intermediate_size, hidden_dim, n_active, n_swapped, n_runs, device)


Profiling Configuration:
  Intermediate size: 8192
  Hidden dim: 3072
  Active indices: 4096 (50.0%)
  Swapped per step: 409 (10.0% of active)
  Number of runs: 100
  Device: cuda

Warming up...


AttributeError: 'str' object has no attribute 'type'